### 3.7.1. Stochastic Programming

$$
\min_{\mathbf{x}}\; \mathbb{E}_{\boldsymbol\xi}\big[\, Q(\mathbf{x},\boldsymbol\xi)\,\big]
= \sum_{s=1}^{S} p_s\, Q(\mathbf{x},\boldsymbol\xi_s),
$$
with scenarios $\boldsymbol\xi_s$ of probability $p_s$.

**Explanation:**

Stochastic programming optimizes a decision that must be fixed *before* uncertain data $\boldsymbol\xi$ is revealed, minimizing the expected cost taken over the distribution of $\boldsymbol\xi$. When the distribution is represented by a finite set of scenarios, the expectation becomes a probability-weighted sum, and the resulting *deterministic equivalent* is an ordinary (often large) [linear or convex program](../04_Convex_Problem_Classes/01_linear_programming.ipynb). The newsvendor problem is the canonical single-stage example: order a quantity now, then pay holding or shortage costs once demand is observed.

**Assumptions:**
- The scenario probabilities $p_s\ge 0$ sum to one.
- The recourse cost $Q(\mathbf{x},\boldsymbol\xi_s)$ is convex in $\mathbf{x}$.

**Numerical Example:**

Order quantity $q$ against demand scenarios $d\in\{80,100,120\}$ with probabilities $\{0.3,0.4,0.3\}$, holding cost $h=1$ per leftover unit and shortage cost $b=4$ per unmet unit:
$$
Q(q,d) = h\,\max(q-d,0) + b\,\max(d-q,0).
$$

Evaluating expected cost at the scenario order levels:
$$
q=100:\ \mathbb{E}[Q]=0.3(20)+0.4(0)+0.3(80)=30,\qquad
q=120:\ \mathbb{E}[Q]=0.3(40)+0.4(20)+0.3(0)=20.
$$
The optimum is $q^\star=120$, matching the critical-ratio rule: the $\tfrac{b}{b+h}=0.8$ quantile of demand is $120$.

In [1]:
demand_scenarios = [80, 100, 120]
scenario_probabilities = [0.3, 0.4, 0.3]
holding_cost, shortage_cost = 1, 4

def expected_cost(order_quantity):
    return sum(probability * (holding_cost*max(order_quantity - demand, 0)
                              + shortage_cost*max(demand - order_quantity, 0))
               for demand, probability in zip(demand_scenarios, scenario_probabilities))

costs = {order: expected_cost(order) for order in demand_scenarios}
best_order = min(costs, key=costs.get)

print("expected cost per order level:", costs)
print("optimal order quantity:", best_order, " expected cost:", costs[best_order])

expected cost per order level: {80: 80.0, 100: 30.0, 120: 20.0}
optimal order quantity: 120  expected cost: 20.0


**Equivalent scipy.linprog implementation:**

In [2]:
import numpy as np
from scipy.optimize import linprog

demand_scenarios = [80, 100, 120]
scenario_probabilities = [0.3, 0.4, 0.3]
holding_cost, shortage_cost = 1, 4

cost = [0.0] + [holding_cost*p for p in scenario_probabilities] + [shortage_cost*p for p in scenario_probabilities]
rows, upper = [], []
for index, demand in enumerate(demand_scenarios):
    over = [0.0]*7; over[0] = 1.0; over[1 + index] = -1.0; rows.append(over); upper.append(demand)
    short = [0.0]*7; short[0] = -1.0; short[4 + index] = -1.0; rows.append(short); upper.append(-demand)
result = linprog(cost, A_ub=np.array(rows), b_ub=np.array(upper), bounds=[(0, None)]*7)

print("optimal order quantity:", round(result.x[0], 1), " expected cost:", round(result.fun, 1))

optimal order quantity: 120.0  expected cost: 20.0


![Newsvendor demand density and stocking level](../../Figures/030702_chance_constraint_density.png)

The critical-ratio rule generalizes: the optimal order quantity is the $\tfrac{b}{b+h}$ quantile of the demand distribution, linking single-stage stochastic programming directly to the quantile reformulation used in [chance-constrained optimization](./02_chance_constrained_optimization.ipynb).

**References:**

[📘 Birge, J. R., & Louveaux, F. (2011). *Introduction to Stochastic Programming* (2nd ed.). Springer.](https://doi.org/10.1007/978-1-4614-0237-4)  
[📗 Postek, K., & Zocca, A. (2024). *Hands-On Mathematical Optimization with Python*, Ch. 8–9. Cambridge University Press.](https://mobook.github.io/MO-book/intro.html)

---

[⬅️ Previous: Combinatorial Optimization](../06_Discrete_and_Combinatorial_Optimization/06_combinatorial_optimization.ipynb) | [Next: Chance-Constrained Optimization ➡️](./02_chance_constrained_optimization.ipynb)